# Budget and Latency Simulation

This notebook simulates many workloads and compares two policies:

- `local_only`: always run on the anchored local machine.
- `burst_when_worth_it`: use a simple heuristic to burst to cloud providers when the latency gain is worth the extra spend and transfer cost.

The purpose is to make tradeoffs legible before any real control plane exists.

In [ ]:
from dataclasses import dataclass
from random import Random
from statistics import mean

rng = Random(7)

@dataclass(frozen=True)
class Provider:
    name: str
    hourly_cost: float
    speed_factor: float
    queue_minutes: float
    transfer_penalty: float

@dataclass(frozen=True)
class Task:
    base_minutes_local: float
    working_set_gb: float
    requires_gpu: bool
    budget: float

providers = [
    Provider("local", 0.55, 1.0, 0.0, 0.0),
    Provider("cpu-cloud", 0.72, 1.8, 1.5, 0.25),
    Provider("gpu-cloud", 2.05, 4.5, 3.0, 0.18),
    Provider("spot-cloud", 0.95, 2.7, 5.0, 0.35),
]

def make_task() -> Task:
    return Task(
        base_minutes_local=rng.uniform(15, 240),
        working_set_gb=rng.uniform(1, 24),
        requires_gpu=rng.random() < 0.35,
        budget=rng.uniform(1.0, 9.0),
    )

In [ ]:
def runtime_minutes(task: Task, provider: Provider) -> float:
    return task.base_minutes_local / provider.speed_factor + provider.queue_minutes + provider.transfer_penalty * task.working_set_gb

def task_cost(task: Task, provider: Provider) -> float:
    return runtime_minutes(task, provider) / 60 * provider.hourly_cost

def policy_local_only(task: Task) -> Provider:
    return providers[0]

def burst_value(task: Task, local: Provider, provider: Provider, multiplier: float) -> float:
    gain = runtime_minutes(task, local) - runtime_minutes(task, provider)
    premium = task_cost(task, provider) - task_cost(task, local)
    return gain - multiplier * max(0.0, premium)

def policy_burst_when_worth_it(task: Task) -> Provider:
    local = providers[0]
    best = local
    best_value = 0.0
    for provider in providers[1:]:
        if task.requires_gpu and provider.name == "cpu-cloud":
            continue
        cost = task_cost(task, provider)
        if cost > task.budget:
            continue
        value = burst_value(task, local, provider, multiplier=128)
        if value > best_value:
            best = provider
            best_value = value
    return best

In [ ]:
tasks = [make_task() for _ in range(250)]

def evaluate_policy(name: str, chooser):
    runs = []
    for task in tasks:
        provider = chooser(task)
        runs.append({
            "provider": provider.name,
            "minutes": runtime_minutes(task, provider),
            "cost": task_cost(task, provider),
        })
    return {
        "policy": name,
        "avg_minutes": round(mean(run["minutes"] for run in runs), 2),
        "avg_cost": round(mean(run["cost"] for run in runs), 2),
        "burst_share": round(sum(run["provider"] != "local" for run in runs) / len(runs), 2),
        "provider_mix": {
            provider.name: sum(run["provider"] == provider.name for run in runs)
            for provider in providers
        },
    }

local_summary = evaluate_policy("local_only", policy_local_only)
burst_summary = evaluate_policy("burst_when_worth_it", policy_burst_when_worth_it)
local_summary, burst_summary

In [ ]:
def chooser_with_penalty(multiplier: float):
    def choose(task: Task) -> Provider:
        local = providers[0]
        best = local
        best_value = 0.0
        for provider in providers[1:]:
            if task.requires_gpu and provider.name == "cpu-cloud":
                continue
            cost = task_cost(task, provider)
            if cost > task.budget:
                continue
            value = burst_value(task, local, provider, multiplier)
            if value > best_value:
                best = provider
                best_value = value
        return best
    return choose

for multiplier in [8, 32, 128, 256, 512]:
    summary = evaluate_policy(f"premium_x_{multiplier}", chooser_with_penalty(multiplier))
    print(summary)

## Takeaway

A burst-aware policy can reduce average latency, but the amount of gain depends heavily on how aggressively the scheduler penalizes cloud premium. That is exactly the policy surface this repo needs notebooks for: not just architecture prose, but tunable experiments.